# 03 · Episodic META-DDIE protocol (Tables 7.12 and 7.13)

Source: `metaddie_clean_run_CLAUDE` (cells 4–22).
1. META-DDIE graph: classes with ≥ 2 examples, 70/15/15 split over **common** classes (1–175) only.
2. Pretrain on common classes (hidden width 384, normalised LDAM).
3. 1-shot episodes, 5- and 10-way, on common-test / fewer / rare pools: nearest prototype and linear head → **Table 7.12**.
4. T3: ChemBERTa-only embeddings (no GNN).
5. T4: inference-time smoothing of the frozen embeddings → **Table 7.13**.

Known issue kept: episode seeds use `hash(split_name) % 7`, which Python randomises per process. To fix the episode draws across sessions, set `PYTHONHASHSEED` before the kernel starts (locally: `PYTHONHASHSEED=0 jupyter lab`).
On the test subgraph the fewer and rare pools usually have too few classes; those rows are skipped.

In [ ]:
DATA_ROOT = "<DATA_ROOT: e.g. /home/you/ddi_data>"   # PLACEHOLDER: thesis inputs (read only)
WORK_ROOT = "<WORK_ROOT: e.g. /home/you/ddi_runs>"   # PLACEHOLDER: new outputs
TEST_MODE = True     # True: small test subgraph from notebook 00, few epochs
                     # False: full thesis graph (needs about 64 GB of RAM)
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [ ]:
import os, sys
# The package is installed in the venv by setup/setup_local.sh (editable install).
# Run this notebook with the "ddi (5090)" kernel.
try:
    import ddi
except ImportError:
    raise SystemExit('ddi is not importable: run "bash setup/setup_local.sh" and select '
                     'the "ddi (5090)" kernel.')
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(ddi.__file__), "..", ".."))
print("python:", sys.executable)
print("ddi from:", os.path.dirname(ddi.__file__))

In [ ]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT)
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

In [ ]:
from ddi import graph as G
from ddi.experiments import episodic as E
from ddi.splits import build_metaddie_graph, save_split, check_split

if TEST_MODE:
    drug_graph, line_graph = G.load_graph_pair(os.path.join(paths.test_graph_dir, f"test_pair_{N_TEST_DRUGS}.bin"))
else:
    G.ensure_graph_bin(paths.graph_zip, paths.graph_bin)
    drug_graph, line_graph = G.load_graph_pair(paths.graph_bin)
line_graph = G.attach_endpoint_features(drug_graph, line_graph)
print(f"line graph: {line_graph.num_nodes()} nodes, {line_graph.num_edges()} edges")

SEED = 42
E.set_seed(SEED)
g = build_metaddie_graph(line_graph, seed=SEED)
del line_graph; gc.collect()
RUN_TAG = f"test{N_TEST_DRUGS}" if TEST_MODE else "full"
split_path = os.path.join(paths.episodic_dir, f"split_{RUN_TAG}.pt")
check_split(g, split_path) if os.path.exists(split_path) else save_split(g, split_path, seed=SEED)

## Pretrain on common classes
`USE_THESIS_CHECKPOINT = True` loads `thesis_metaddie_clean_run/pretrain_common_only.pt` instead (full graph only).

In [ ]:
USE_THESIS_CHECKPOINT = False
if USE_THESIS_CHECKPOINT:
    assert not TEST_MODE
    model = E.load_pretrained(g, os.path.join(paths.thesis_episodic_dir, "pretrain_common_only.pt"), device)
else:
    ckpt = os.path.join(paths.episodic_dir, f"pretrain_common_only_{RUN_TAG}.pt")
    model = E.train_common_only(g, device, ckpt, epochs=(2 if TEST_MODE else 100), batch_size=512,
                                fanouts=(15, 10), monitor="macro_f1", patience=20)

## Table 7.12: episodic accuracy

In [ ]:
EPISODES = 200 if TEST_MODE else 10_000
Z = E.compute_all_embeddings(model, g, device)
labels_all_dev, pools = E.make_pools(g, device)
for name, pool in pools.items():
    for N in (5, 10):
        try:
            print(f"  {name:6s} N={N:<3d}: {len(E.eligible(pool, N, 1))} eligible classes")
        except ValueError as err:
            print(f"  {name:6s} N={N:<3d}: INSUFFICIENT ({err})")
results = E.run_episodic_suite(Z, labels_all_dev, pools, EPISODES, SEED)
json.dump(E.results_to_json(results), open(os.path.join(paths.episodic_dir, f"episodic_results_{RUN_TAG}.json"), "w"), indent=2)
print("META-DDIE paper:", E.METADDIE_PAPER)

## T3: ChemBERTa-only baseline

In [ ]:
t3 = {}
for mode in ["mean", "hadamard", "set_untrained"]:
    print(f"\n=== ChemBERTa-only, fusion = {mode}")
    Zc = E.build_chemberta_only_embeddings(g, device, fusion_mode=mode, seed=SEED)
    r = E.run_episodic_suite(Zc, labels_all_dev, pools, EPISODES, SEED, seed_salt=mode)
    t3.update({(mode,) + k: v for k, v in r.items()})
    del Zc; torch.cuda.empty_cache()
json.dump(E.results_to_json(t3), open(os.path.join(paths.episodic_dir, f"t3_chemberta_only_{RUN_TAG}.json"), "w"), indent=2)

## T4: inference-time smoothing (Table 7.13)

In [ ]:
E.set_seed(SEED)
train_nids = g.ndata["train_mask"].nonzero(as_tuple=False).squeeze().to(device)
knn_idx, knn_sim = E.build_knn_to_train(Z, train_nids, k=5)
apply_mask = (((labels_all_dev >= E.FEWER_MIN) & (labels_all_dev <= E.FEWER_MAX))
              | ((labels_all_dev >= E.RARE_MIN) & (labels_all_dev <= E.RARE_MAX)))
print(f"smoothing applies to {int(apply_mask.sum())} nodes (fewer + rare)")
t4 = {}
for cfg, mode, params in E.T4_FUSION_CONFIGS:
    print(f"\n=== smoothing {cfg}")
    Zs = E.compute_smoothed_Z(Z, knn_idx, knn_sim, apply_mask, fusion_mode=mode, fusion_params=params)
    r = E.run_episodic_suite(Zs, labels_all_dev, pools, EPISODES, SEED, seed_salt=cfg)
    t4.update({(cfg,) + k: v for k, v in r.items()})
    del Zs; torch.cuda.empty_cache()
json.dump(E.results_to_json(t4), open(os.path.join(paths.episodic_dir, f"t4_smoothing_{RUN_TAG}.json"), "w"), indent=2)